# 11 · Calibrated error bars: conformal prediction (model H)

*Level: advanced (but short). Guide Section 15. Code: `calibrate.Conformal`, `calibrate.conformal_factor`.*

**Idea.** Measure, on packets the model did not train on, how many of its own error bars it takes to cover the truth. Then rescale the error bars by
that factor. It works for **any** model.

In [1]:
from tutorial_utils import *

from ml.calibrate import Conformal, conformal_factor

## 1. The recipe, by hand

1. For held-out examples: $z_i = |y_i - \mu_i|/s_i$, the error in units of the model's own standard deviation.
2. The factor for coverage $c$ is the $\lceil (n+1)c\rceil$-th smallest $z_i$.
3. Predict with $\mu \pm q_c\, s$.

Below, a deliberately **over-confident** model claims σ = 0.5 when the truth scatters by 1.

In [2]:
rng = np.random.default_rng(0)
cal_err, test_err, claimed = rng.normal(0, 1, 200), rng.normal(0, 1, 5000), 0.5
z = np.abs(cal_err) / claimed
n = len(z); k = int(np.ceil((n + 1) * 0.68)) - 1
q_hand = np.sort(z)[k]
print(f"factor by hand {q_hand:.3f}; calibrate.conformal_factor {conformal_factor(z, 0.68):.3f}  (ideal 1/0.5 = 2)")
print(f"coverage on new data: raw {np.mean(np.abs(test_err) <= claimed):.2f}, calibrated {np.mean(np.abs(test_err) <= q_hand * claimed):.2f} (target 0.68)")

factor by hand 2.084; calibrate.conformal_factor 2.084  (ideal 1/0.5 = 2)
coverage on new data: raw 0.37, calibrated 0.70 (target 0.68)


**The guarantee.** If new examples are **exchangeable** with the calibration ones (they could have been any of them), the coverage is at least
$c$, whatever the model. Our version uses 5 packet-grouped folds, so every training packet serves for calibration.

## 2. On the toy CCD: boosted trees and ridge, raw and calibrated

In [3]:
dfa, ca, Ya, ga = toy_scan(120, seed=1); dfb, cb, Yb, gb = toy_scan(60, seed=2)
Xa, _ = data.make_inputs(dfa, ("theta", "trace", "ref"), drop_constant=False)
Xb, _ = data.make_inputs(dfb, ("theta", "trace", "ref"), drop_constant=False)
for name, make in (("boosted trees", lambda: models.GBT()), ("Bayesian ridge", lambda: models.BayesRidge())):
    m, s = make().fit(Xa, Ya).predict(Xb); _ = report(f"{name}, raw", Yb, m, s)
    cal = Conformal(make).fit(Xa, Ya, ga); m, s = cal.predict(Xb); _ = report(f"{name}, conformal", Yb, m, s)
    print("   factors (68 %):", cal.factors[0.68].round(2))

boosted trees, raw                       sigma_e:  16.2 % (cover 0.43)   lambda:  28.5 % (cover 0.24)   gain:   9.0 % (cover 0.54)


boosted trees, conformal                 sigma_e:  16.2 % (cover 0.62)   lambda:  28.5 % (cover 0.60)   gain:   9.0 % (cover 0.62)
   factors (68 %): [1.91 3.04 1.27]
Bayesian ridge, raw                      sigma_e:  10.5 % (cover 0.83)   lambda:  11.6 % (cover 0.66)   gain:   0.9 % (cover 0.64)
Bayesian ridge, conformal                sigma_e:  10.5 % (cover 0.77)   lambda:  11.6 % (cover 0.64)   gain:   0.9 % (cover 0.62)
   factors (68 %): [0.89 0.97 0.97]


## 3. What it cannot do: a new situation

Calibrated on module 100 and tested on a module never seen, the coverage is no longer guaranteed: the new module is **not exchangeable** with the old one.

In [4]:
tr, _ = synthetic.make_dataset(120, seed=1, n_boards=2); te, _ = synthetic.make_dataset(40, seed=99, n_boards=2)
tr = tr[tr.board == 100].reset_index(drop=True); te_old, te_new = te[te.board == 100], te[te.board == 101]
Xt, _ = data.make_inputs(tr, ("theta", "trace", "ref"), drop_constant=False)
cal = Conformal(lambda: models.GBT()).fit(Xt, data.make_targets(tr)[0], data.groups_of(tr))
for label, d in (("same module", te_old), ("NEW module", te_new)):
    X_, _ = data.make_inputs(d, ("theta", "trace", "ref"), drop_constant=False); m, s = cal.predict(X_)
    _ = report(f"conformal trees, {label}", data.make_targets(d)[0], m, s)

conformal trees, same module             sigma_e:  20.1 % (cover 0.59)   lambda:  14.5 % (cover 0.79)   gain:   9.6 % (cover 0.70)
conformal trees, NEW module              sigma_e:  58.5 % (cover 0.24)   lambda:  16.3 % (cover 0.89)   gain:  28.1 % (cover 0.00)


## When to use it

Always, as the last step: it costs 5 extra fits. It is essential for boosted trees and a check for every other model. Remember the
exchangeability condition.

Next: **12 · Pre-training on unlabelled traces (D).**